# Olist Dataset Profiling

Lightweight exploration for the raw Kaggle dataset `olistbr/brazilian-ecommerce`.

This notebook is exploratory only. Bronze remains raw, and reusable pipeline logic belongs under `src/`.

## Project Setup

The path setup below supports launching this notebook from the repository root, the project root, or the `notebooks/` directory.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "src" / "ingestion").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "src" / "ingestion").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd / "projects" / "03-retail-revenue-analytics" / "src" / "ingestion").exists():
    PROJECT_ROOT = cwd / "projects" / "03-retail-revenue-analytics"
else:
    raise RuntimeError("Launch this notebook from the project directory, notebook directory, or repository root.")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ingestion.raw_inventory import discover_raw_files, get_raw_data_dir, list_supported_data_files
from processing.bronze.profiling import select_main_csv_file

raw_dir = get_raw_data_dir()
raw_dir

WindowsPath('C:/Users/willi/OneDrive/Área de Trabalho/PhaifferTech/repos/data-engineering-portfolio/projects/03-retail-revenue-analytics/data/bronze/raw/olist_brazilian_ecommerce')

## Raw Directory

The directory below is the raw Bronze landing area. It should contain files downloaded by the ingestion job.

In [2]:
raw_dir.exists(), raw_dir

(True,
 WindowsPath('C:/Users/willi/OneDrive/Área de Trabalho/PhaifferTech/repos/data-engineering-portfolio/projects/03-retail-revenue-analytics/data/bronze/raw/olist_brazilian_ecommerce'))

## Raw File Inventory

Bronze inventories all landed raw files. Hidden Kaggle operational artifacts can appear here and are tracked for transparency.

In [3]:
inventory = discover_raw_files(raw_dir)
inventory

{'raw_directory': 'data/bronze/raw/olist_brazilian_ecommerce',
 'supported_extensions': ['.csv'],
 'file_count': 10,
 'supported_data_file_count': 9,
 'files': [{'path': 'data/bronze/raw/olist_brazilian_ecommerce/.complete/datasets/olistbr/brazilian-ecommerce/2/bundle.complete',
   'name': 'bundle.complete',
   'extension': '.complete',
   'size_bytes': 0,
   'is_supported_data_file': False,
   'is_selected_for_bronze_profiling': False},
  {'path': 'data/bronze/raw/olist_brazilian_ecommerce/olist_customers_dataset.csv',
   'name': 'olist_customers_dataset.csv',
   'extension': '.csv',
   'size_bytes': 9033957,
   'is_supported_data_file': True,
   'is_selected_for_bronze_profiling': True},
  {'path': 'data/bronze/raw/olist_brazilian_ecommerce/olist_geolocation_dataset.csv',
   'name': 'olist_geolocation_dataset.csv',
   'extension': '.csv',
   'size_bytes': 61273883,
   'is_supported_data_file': True,
   'is_selected_for_bronze_profiling': True},
  {'path': 'data/bronze/raw/olist_brazi

## Supported CSV Files

Bronze v1 profiles only supported `.csv` files and excludes hidden operational paths from profiling.

In [4]:
csv_files = list_supported_data_files(raw_dir)
csv_files

[WindowsPath('C:/Users/willi/OneDrive/Área de Trabalho/PhaifferTech/repos/data-engineering-portfolio/projects/03-retail-revenue-analytics/data/bronze/raw/olist_brazilian_ecommerce/olist_customers_dataset.csv'),
 WindowsPath('C:/Users/willi/OneDrive/Área de Trabalho/PhaifferTech/repos/data-engineering-portfolio/projects/03-retail-revenue-analytics/data/bronze/raw/olist_brazilian_ecommerce/olist_geolocation_dataset.csv'),
 WindowsPath('C:/Users/willi/OneDrive/Área de Trabalho/PhaifferTech/repos/data-engineering-portfolio/projects/03-retail-revenue-analytics/data/bronze/raw/olist_brazilian_ecommerce/olist_order_items_dataset.csv'),
 WindowsPath('C:/Users/willi/OneDrive/Área de Trabalho/PhaifferTech/repos/data-engineering-portfolio/projects/03-retail-revenue-analytics/data/bronze/raw/olist_brazilian_ecommerce/olist_order_payments_dataset.csv'),
 WindowsPath('C:/Users/willi/OneDrive/Área de Trabalho/PhaifferTech/repos/data-engineering-portfolio/projects/03-retail-revenue-analytics/data/bron

## Selected Main CSV

The selected file is the largest supported CSV by size, with path-based tie breaking. This is a raw-stage profiling rule only, not a final business-grain decision.

In [5]:
selected_csv = select_main_csv_file(csv_files)
selected_csv

WindowsPath('C:/Users/willi/OneDrive/Área de Trabalho/PhaifferTech/repos/data-engineering-portfolio/projects/03-retail-revenue-analytics/data/bronze/raw/olist_brazilian_ecommerce/olist_geolocation_dataset.csv')

## DataFrame Preview

Read the selected raw CSV for exploratory inspection only. No pipeline logic is defined in this notebook.

In [6]:
import pandas as pd

df = pd.read_csv(selected_csv)
df.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [7]:
df.shape

(1000163, 5)

In [8]:
list(df.columns)

['geolocation_zip_code_prefix',
 'geolocation_lat',
 'geolocation_lng',
 'geolocation_city',
 'geolocation_state']

## Pandas Info

Pandas inference is useful for first-pass inspection, but business type decisions are deferred to future Silver contracts.

In [9]:
from io import StringIO

buffer = StringIO()
df.info(buf=buffer)
print(buffer.getvalue())

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 50.1 MB



In [10]:
df.isna().sum().sort_values(ascending=False)

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [11]:
int(df.duplicated().sum())

261831

## Scope Reminder

This notebook supports source understanding. It does not define Silver tables, facts, dimensions, revenue KPIs, DBT marts, orchestration, APIs, or dashboards.